# SDA-DSC-212 — Time Series Analysis and Forecasting
## Lab 3 — Classical Baselines: SARIMAX and ETS

**Duration** 50 minutes · **Work in pairs** · `git checkout lab3-solution`

> The one rule for this whole course: every feature, every split and every scaler
> must be computable using only data that existed at the forecast origin.

### Tasks

| min | task |
|---|---|
| 5 | Aggregate the hourly series to **daily max** — the reserve-margin question is about the peak. Restrict to 2022 onward, build the exogenous frame (`cdd`, `hdd`, `is_weekend`, `is_ramadan`, `is_eid`), hold out the last 28 days. |
| 12 | Fit SARIMAX `(2,1,1)(1,1,1)[7]`. Explain why `m=7` on a daily series rather than `m=24` on the hourly one. Run `residual_diagnostics` and report Ljung-Box honestly. |
| 10 | Fit ETS with `fit_ets` (damped trend, `seasonal_periods=7`). Sweep the taxonomy with `ets_grid` by AIC. State when AIC comparisons are valid. |
| 10 | Forecast 28 days from both with 90% intervals. SARIMAX needs the **future** exogenous rows. Plot both forecasts against the held-out actuals. |
| 8 | MAE / MAPE / MASE and 90% coverage for each. Fit a no-exogenous SARIMA to price what temperature is worth. Compute the seasonal-naive baseline. |
| 5 | Commit. |

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.figsize": (11, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25})

DATA = "../data"

Lab 1's output, reloaded — the labs are independent notebooks.

In [ ]:
from tayyar.data.load import load_demand, load_calendar, join_calendar

df = load_demand(f"{DATA}/ksa_grid_demand.csv")
df = join_calendar(df, load_calendar(f"{DATA}/ksa_calendar.csv"))
y = df["demand_mw"].astype(float)

print(f"hourly rows: {len(y):,}   from {y.index[0]} to {y.index[-1]}")

### Task 1 — the daily-max series, the exogenous frame, and the hold-out

Two decisions here, both of which encode the question rather than the data.

**Daily *max*, not daily mean.** The deliverable is a reserve margin. A control room sizes
reserve against the peak the system must carry, not against the average it happens to run at.
Aggregating with `.mean()` would answer a question nobody asked, and would answer it about
5,000 MW too low.

**`m = 7` on a daily series, not `m = 24` on the hourly one.** This is a wall you should walk
into deliberately. SARIMA's state space grows with the seasonal period: at `m=24` the
state vector is large enough that estimation slows from seconds to many minutes, and
`auto_arima` with `m=24` may simply never converge on this series. The classical route
therefore models the *daily* series with a weekly cycle, `m=7`, and leaves the sub-daily
structure to the feature-based ML route in Module 4. That hand-off is not an admission of
defeat; it is the honest boundary of the classical toolkit, and knowing where it sits is
part of the learning outcome.

In [ ]:
d_daily = y.resample("D").max()          # the PEAK, because the question is reserve margin
d_daily = d_daily["2022":]                # two years is plenty for a daily model

exog = pd.DataFrame({
    "cdd":        np.clip(df["temp_c"].resample("D").max() - 21.0, 0, None),
    "hdd":        np.clip(14.0 - df["temp_c"].resample("D").min(), 0, None),
    "is_weekend": df["is_weekend"].resample("D").max().astype(int),
    "is_ramadan": df["is_ramadan"].resample("D").max().astype(int),
    "is_eid":     df["is_eid"].resample("D").max().astype(int),
})

H = 28
tr, te = d_daily[:-H], d_daily[-H:]
Xtr, Xte = exog.loc[tr.index], exog.loc[te.index]

print(f"daily-max series : {len(d_daily)} days, {d_daily.index[0].date()} to {d_daily.index[-1].date()}")
print(f"train            : {len(tr)} days  (to {tr.index[-1].date()})")
print(f"hold-out         : {len(te)} days  (from {te.index[0].date()})")
print(f"exog columns     : {list(Xtr.columns)}")
print()
print(Xtr.tail(3).round(2).to_string())

**Expected**

```
daily-max series : 730 days, 2022-01-01 to 2023-12-31
train            : 702 days  (to 2023-12-03)
hold-out         : 28 days  (from 2023-12-04)
exog columns     : ['cdd', 'hdd', 'is_weekend', 'is_ramadan', 'is_eid']
```

### Task 2 — SARIMAX (2,1,1)(1,1,1)[7], and an honest residual check

In [ ]:
from tayyar.models.sarimax import fit_sarimax, forecast_sarimax, residual_diagnostics

sar = fit_sarimax(tr, Xtr, order=(2, 1, 1), seasonal_order=(1, 1, 1, 7))

print("exogenous coefficients (MW per unit):")
print(sar.params[list(Xtr.columns)].round(1).to_string())
print()

diag = residual_diagnostics(sar, lags=14)
for k, v in diag.items():
    print(f"{k:<26} {v}")

**Expected**

```
exogenous coefficients (MW per unit):
cdd            653.6
hdd             52.7
is_weekend       0.8
is_ramadan    1129.9
is_eid       -4139.4

ljung_box_p                0.0
residual_autocorrelation   True
verdict                    UNDER-SPECIFIED: structure remains in residuals
jarque_bera_p              0.0
aic                        11443.8
bic                        11493.6
```

The coefficients read out loud without translation: every cooling-degree-day above 21 °C adds
about **654 MW** to the daily peak; Ramadan adds about **1,130 MW**; Eid takes about
**4,139 MW** off. That sentence is the reason classical models survive in control rooms.

Now the uncomfortable part, and we are going to state it plainly rather than bury it.
**Ljung-Box p = 0.00.** The null is "the residuals are independent", and we reject it. Our own
model, fitted with the orders this course taught you to read off a correlogram, leaves
autocorrelation in its residuals. By the standard this course set in Module 3, the model is
**under-specified**. Jarque-Bera also rejects normality, which means the 90% interval is an
approximation resting on an assumption the residuals do not honour.

We are not going to pretend otherwise, and we are not going to grind through order after order
to chase the p-value above 0.05. The honest reading is that a daily SARIMAX with five
regressors cannot capture everything driving this series — the sub-daily shape, the nonlinear
temperature response, the interaction between the two. That is precisely the gap the
feature-based ML route in Module 4 exists to fill. A model that fails its own diagnostic and
says so is worth more than one that passes because nobody ran the test.

In [ ]:
fig = sar.plot_diagnostics(figsize=(11, 6), lags=28)
plt.tight_layout()

### Task 3 — ETS, and the rules for reading an AIC

In [ ]:
from tayyar.models.ets import fit_ets, forecast_ets, ets_grid

ets = fit_ets(tr, seasonal_periods=7)      # trend="add", damped=True, seasonal="add"

par = {k: v for k, v in ets.params.items() if np.isscalar(v)}
for k in ("smoothing_level", "smoothing_trend", "smoothing_seasonal", "damping_trend"):
    print(f"{k:<20} {par[k]:.4f}")
print(f"{'aic':<20} {ets.aic:.1f}")
print()

print(ets_grid(tr, seasonal_periods=7).to_string(index=False))

**Expected**

```
smoothing_level      0.2638
smoothing_trend      0.0260
smoothing_seasonal   0.0000
damping_trend        0.9912
aic                  10532.0

trend seasonal  damped     aic
  add      mul    True 10511.6
  add      mul   False 10511.7
  NaN      mul   False 10518.3
  add      add    True 10532.0
  NaN      add   False 10538.6
  add      add   False 10540.9
  add      NaN   False 10898.3
  add      NaN    True 10898.3
  NaN      NaN   False 10914.6
```

`phi = 0.991` — the damping is mild but present, and it is what stops a 28-day horizon from
running off. `gamma = 0.000` says the weekly seasonal factors, once initialised, barely need
updating: the weekly shape of the peak is stable.

The sweep prefers **ETS(A,Ad,M)**, multiplicative seasonality, by about 20 AIC over the
additive fit we are carrying forward. That is a real gap, not noise, and on a series whose
swing grows with its level it is the expected answer. Note what makes the multiplicative fit
*available* at all: the daily-max series contains no zeros. Try the same sweep on the raw
hourly series with its sensor dropouts and every multiplicative row fails.

**When may these AICs be compared?** Only down this column, because every row was fitted to
the same untransformed 702-day target. Never against the SARIMAX AIC above (different model
class, different likelihood), never against a model fitted to `log(y)`, and never across
different `d`.

### Task 4 — 28-day forecasts with 90% intervals

`get_forecast(steps=28, exog=...)` needs **exactly 28 rows of future exogenous values**. That
requirement is not an API inconvenience, it is the definition of an exogenous regressor:
temperature qualifies only because a day-ahead weather forecast exists. If you have to wait
for the realised value, the variable cannot be used, and using the realised value anyway is a
leak that looks like skill — the model gets tomorrow's temperature for free and reports a MAE
no production system could reproduce.

For reproducibility in this lab we pass the realised temperatures, so the numbers here are a
mild optimistic bound: they price what temperature is worth *given a perfect forecast of it*.
`data/temp_forecast_28d.csv` holds the actual day-ahead forecast series; substituting it is
the fast-finisher extension, and the MAE gap you measure is the cost of the weather
forecast's own error propagating into yours.

In [ ]:
f_sar = forecast_sarimax(sar, H, Xte, alpha=0.10)     # 90% interval
f_ets = forecast_ets(ets, H)

fig, ax = plt.subplots(figsize=(11, 4.6))
tr[-60:].plot(ax=ax, color="#7F7F7F", lw=1, label="train (last 60 d)")
te.plot(ax=ax, color="#1F3864", lw=2, label="actual (held out)")
ax.fill_between(f_sar.index, f_sar["lo"], f_sar["hi"], color="#8FAADC", alpha=.45,
                label="SARIMAX 90% interval")
f_sar["yhat"].plot(ax=ax, color="#C00000", lw=1.8, label="SARIMAX + temp + calendar")
f_ets.plot(ax=ax, color="#2E7D32", lw=1.8, ls="--", label="ETS(A,Ad,A)")
ax.set_ylabel("daily peak MW")
ax.set_title("28-day daily-peak forecast against the held-out actuals")
ax.legend(frameon=False, ncols=3, fontsize=8)
plt.tight_layout()

print(pd.DataFrame({"actual": te, "sarimax": f_sar["yhat"].round(0),
                    "ets": f_ets.round(0)}).head(7).to_string())

The SARIMAX track sits on the actuals through the whole window and the band is narrow
enough to be useful. The ETS track runs visibly high — hold that thought until the bias
column in Task 5.

### Task 5 — the scoreboard: accuracy, bias and interval width

In [ ]:
from tayyar.eval.metrics import all_metrics

# What is temperature worth? Fit the same model with no exogenous regressors at all.
sar_noex = fit_sarimax(tr, None, order=(2, 1, 1), seasonal_order=(1, 1, 1, 7))
f_noex = forecast_sarimax(sar_noex, H, None, alpha=0.10)

# And the free baseline everything must beat.
sn = pd.Series([tr.iloc[-7:].to_numpy()[i % 7] for i in range(H)], index=te.index)

rows = {
    "SARIMAX + temp + calendar": f_sar["yhat"],
    "SARIMA (no exog)":          f_noex["yhat"],
    "ETS(A,Ad,A) damped":        f_ets,
    "Seasonal-naive (m=7)":      sn,
}
tbl = pd.DataFrame({k: all_metrics(te, v, tr, 7) for k, v in rows.items()}).T
print(tbl[["MAE", "RMSE", "MAPE_%", "MASE", "bias"]].to_string())
print()

def coverage(f):
    inside = (te.to_numpy() >= f["lo"].to_numpy()) & (te.to_numpy() <= f["hi"].to_numpy())
    return round(float(inside.mean()), 3), round(float((f["hi"] - f["lo"]).mean()), 1)

for name, f in (("SARIMAX + temp + calendar", f_sar), ("SARIMA (no exog)", f_noex)):
    cov, width = coverage(f)
    print(f"{name:<26} 90% coverage {cov:<6} mean interval width {width:>8.1f} MW")
print(f"{'ETS(A,Ad,A) damped':<26} 90% coverage    n/a   "
      f"— the Holt-Winters API returns no analytic interval")

**Expected**

```
                             MAE    RMSE  MAPE_%    MASE   bias
SARIMAX + temp + calendar  666.2   826.4    2.05  0.3325   36.7
SARIMA (no exog)           686.0   843.6    2.14  0.3424  142.3
ETS(A,Ad,A) damped         868.5  1036.5   2.682  0.4335  782.4
Seasonal-naive (m=7)      2609.0  3040.1   8.115  1.3021 2534.0

SARIMAX + temp + calendar  90% coverage 0.964  mean interval width   3875.3 MW
SARIMA (no exog)           90% coverage 1.0    mean interval width   8896.1 MW
ETS(A,Ad,A) damped         90% coverage    n/a — the Holt-Winters API returns no analytic interval
```

**MASE first.** Every model beats the free seasonal-naive baseline (MASE 1.302), and beats it
by a wide margin. That is the only sentence in this table that establishes the models are
worth running at all.

**Now look at the bias column, not the MAE column.** The no-exogenous model runs **+142 MW
high** on average; adding temperature and the calendar pulls that to **+37 MW**. A biased
forecast is not an occasional error, it is the same error every single day: a reserve margin
sized on a forecast that is systematically 142 MW high is 142 MW of capacity held, paid for
and never used, every day of the year. The MAE improvement from temperature is a modest 20 MW;
the bias improvement is 105 MW and matters more to the operator.

**What temperature is actually worth is in the intervals.** Coverage 0.964 versus 1.000 looks
like the no-exog model winning until you read the width beside it: 3,875 MW against
8,896 MW. Both intervals cover; one of them is **2.3 times wider**, and the reserve margin is
set from the upper bound. Knowing tomorrow's temperature halves the width of the band the
control room must hold capacity against. That, not the MAE, is the business case.

**ETS runs +782 MW high** across the window — a damped additive trend fitted to two years that
end in a rising December simply carries that rise forward. It is the fastest model here and it
needs no exogenous data at all, which is exactly why it is the right choice for a thousand
series and the wrong choice for this one.

### Mini-quiz

1. **What do p, d, q control?** The AR order, the differencing order, and the MA order.
2. **`m` for a daily cycle on hourly data?** 24. (And on this lab's *daily* series, the weekly
   cycle gives `m = 7`.)
3. **Why must exogenous regressors be known at forecast time?** Because forecasting `steps`
   ahead requires their future values. If you cannot obtain them ahead of time the regressor is
   simply unavailable; if you substitute the realised value you have built a leak that will
   report skill you cannot reproduce in production.
4. **Ljung-Box p < 0.05?** Residual autocorrelation remains — there is structure the model has
   not captured. The model is under-specified; raise an order, or accept and document the
   limitation as we did above.
5. **When is a damped trend preferable?** Almost always, and especially for multi-step
   horizons. An undamped linear trend extrapolated far enough produces a confidently absurd
   number.

### Task 6 — commit

The deliverable is the comparison table with the bias and interval-width columns intact, plus
one honest paragraph on the Ljung-Box result. Do not drop the failing diagnostic from the
write-up.

```
git add -A && git commit -m "feat(models): SARIMAX + ETS baselines with intervals and diagnostics"
```

### Instructor notes

- The Ljung-Box failure is the centrepiece of this lab, not an embarrassment to manage. Say it
  out loud: the course's own model fails a test the course taught, and the participants found
  it by running the diagnostic they were told to run. Then draw the line straight to Module 4 —
  the ML route exists because this wall is real. A cohort that watches an instructor report a
  failed diagnostic honestly will report their own.
- Branch `sim-aic-trap` compares AIC across models with different `d` and declares the wrong
  winner. Let a pair defend the winner before you intervene. The fix is not a better search;
  it is understanding that likelihoods computed on differently-differenced data are not
  commensurable, so the comparison never had meaning to begin with.
- Branch `sim-nodamp` forecasts 90 days with an undamped Holt trend and arrives at a peak
  demand the Central Operating Area could not physically carry. Ask what the operator would do
  with that number before you show the fix (`damped_trend=True`).
- Run "explain it to the control room" after Task 5, 10 minutes: each pair writes ONE sentence
  explaining the SARIMAX forecast to a shift engineer. Then strike out every sentence
  containing the word "parameter". What survives is usually the coefficient sentence — 654 MW
  per cooling-degree-day, 1,130 MW for Ramadan, −4,139 MW for Eid — which is the point.
- Troubleshooting:

  | symptom | cause | fix |
  |---|---|---|
  | `auto_arima` runs for minutes and never finishes | `m=24` on the hourly series | model the daily series with `m=7`; leave sub-daily to Module 4 |
  | Ljung-Box p < 0.05 | model under-specified | raise an order — and if it still fails, document it rather than hide it |
  | forecast diverges to implausible values | undamped trend | `damped_trend=True` |
  | `ConvergenceWarning: Maximum Likelihood optimization failed to converge` | the (2,1,1)(1,1,1)[7] likelihood is flat in places | expected on this series; the fit is usable, but it is a second hint the specification is not ideal |
  | `get_forecast` raises on the exog | wrong number of future rows | supply exactly `steps` rows of future exog, indexed to the forecast dates |
  | multiplicative ETS fails to fit | the series contains a zero | multiplicative seasonality needs strictly positive data — the hourly series has sensor zeros, the daily max does not |

- Fast finishers: rebuild `Xte` from `data/temp_forecast_28d.csv` instead of the realised
  temperatures and re-measure. The MAE gap between the two runs is the cost of the weather
  forecast's own error propagating into the load forecast — the number the case study asks
  them to widen the interval for.